# Automated Quality Control with Transfer Learning – Manufacturing Defect Detection
### Discovery-to-Action (DTA) Framework Implementation Report

In [ ]:
# =====================================================================
# TRANSFER LEARNING MANUFACTURING INSPECTION PIPELINE
# =====================================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Sequential
from sklearn.metrics import classification_report, confusion_matrix

print("📥 Simulating Manufacturing Image Datasets (Defective vs Normal)...")
# Setup fake directory matrix structures on disk to satisfy generator constraints
os.makedirs('data/train/defective', exist_ok=True)
os.makedirs('data/train/normal', exist_ok=True)
os.makedirs('data/test/defective', exist_ok=True)
os.makedirs('data/test/normal', exist_ok=True)

# Generate dummy structural image arrays to build clean generators
for i in range(10):
    plt.imsave(f'data/train/defective/defect_{i}.png', np.random.rand(224, 224, 3))
    plt.imsave(f'data/train/normal/normal_{i}.png', np.random.rand(224, 224, 3))
    plt.imsave(f'data/test/defective/defect_{i}.png', np.random.rand(224, 224, 3))
    plt.imsave(f'data/test/normal/normal_{i}.png', np.random.rand(224, 224, 3))
print("✅ Local Image Directories Structured.")

## 🛠️ Discovery Phase: Data Augmentation & Preprocessing
We use `ImageDataGenerator` configured with ResNet50's native preprocessing to scale and augment incoming image layers.

In [ ]:
BATCH_SIZE = 2
IMAGE_SIZE = (224, 224)

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    zoom_range=0.15,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_directory(
    'data/train', target_size=IMAGE_SIZE, batch_size=BATCH_SIZE, class_mode='binary', shuffle=True
)

test_generator = test_datagen.flow_from_directory(
    'data/test', target_size=IMAGE_SIZE, batch_size=BATCH_SIZE, class_mode='binary', shuffle=False
)

## 🧬 Technical Phase: Custom Classification Head Design & Feature Freezing
We isolate the pre-trained weights of ResNet50 and strip away the dense top layer using `include_top=False`.

In [ ]:
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False  # Freeze all feature extraction weights

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
print("🏋️‍♂️ Fitting Custom Prediction Heads on Factory Data...")
history = model.fit(train_generator, validation_data=test_generator, epochs=3, verbose=1)
print("✅ Optimization complete.")

## 📊 Evaluation Pass: Precision, Recall & Reports

In [ ]:
predictions = model.predict(test_generator)
y_pred = (predictions > 0.5).astype(int).flatten()
y_true = test_generator.classes

print("=== 📈 PRODUCTION EVALUATION REPORT ===")
print(classification_report(y_true, y_pred, target_names=['Defective', 'Normal']))

## 🚨 Action Phase: Automation Inference Routing
Simulating real-world deployment where defect probability scores dictate factory robotics responses.

In [ ]:
sample_image = np.random.rand(1, 224, 224, 3)
defect_probability = model.predict(sample_image)[0][0]

print(f"🔍 Quality Control Check Output Probability: {defect_probability:.4f}")
THRESHOLD = 0.85

if defect_probability >= THRESHOLD:
    print(f"🚨 ACTION REQUIRED: Model confidence ({defect_probability:.2%}) >= {THRESHOLD:.2%}.")
    print("🤖 COMMAND SENT: Signaling robotic sorting arm to route part to the REJECT ESCALATION BIN.")
else:
    print(f"✅ PASS: Model confidence ({defect_probability:.2%}) below failure threshold.")
    print("📦 COMMAND SENT: Part approved. Advancing item down primary packaging line assembly.")